# Etapa 2: Investigação e Seleção dos dados.

> Nosso objetivo na Etapa 2 é investigar as bases analíticas e selecionar os dados que serão usados na modelagem.

## Passo 2.1: Identificação das bases disponíveis.

In [ ]:
from pathlib import  Path

import pandas as pd

DATA_DIR =  Path("../../data")

pastas = [
    DATA_DIR / "processed",
    DATA_DIR / "final"
]

arquivos = [
    "dataset_vocabulario.csv",
    "dataset_tde.csv",
    "dim_aluno.csv",
    "pareado_vocabulario.csv",
    "pareado_tde.csv"
]

bases = {}

for arquivo in arquivos:
    encontrado = False

    for pasta in pastas:
        caminho = pasta / arquivo
        if caminho.exists():
            nome = caminho.stem
            bases[nome] = pd.read_csv(caminho)
            print(f"{arquivo}: carregado de {pasta.name}")
            encontrado = True
            break
    
    if not encontrado:
        print(f"{arquivo}: Não encontrado")

## Passo 2.2: Investigação da Estrutura dos Dados.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data")

pastas = [
    DATA_DIR / "processed",
    DATA_DIR / "final"
]

arquivos = [
    "dataset_vocabulario.csv",
    "dataset_tde.csv",
    "dim_aluno.csv",
    "pareado_vocabulario.csv",
    "pareado_tde.csv"
]

bases = {}

for arquivo in arquivos:
    encontrado = False

    for pasta in pastas:
        caminho = pasta / arquivo
        if caminho.exists():
            nome = caminho.stem
            bases[nome] = pd.read_csv(caminho)
            print(f"{arquivo}: carregado de {pasta.name}")
            encontrado = True
            break

    if not encontrado:
        print(f"{arquivo}: Não encontrado")

print("\nBases carregadas:", list(bases.keys()))

for nome, df in bases.items():
    print(f"\n{'=' * 50}")
    print(f"BASE: {nome}")
    print(f"{'=' * 50}")

    print("Dimensões:", df.shape)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos de dados:")
    print(df.dtypes)

    print("\nPrimeiros registros:")
    display(df.head())

for nome, df in bases.items():
    print(f"\nBASE: {nome}")

    ausentes = df.isna().sum()

    print(
        pd.DataFrame({
            "ausentes": ausentes,
            "percentual": (ausentes / len(df) * 100).round(2)
        })
    )

## Passo 2.3: Investigação dos Registros Pareados.

In [ ]:
for nome in ["pareado_vocabulario", "pareado_tde"]:

    if nome not in bases:
        continue

    df = bases[nome]

    print(f"\nBASE: {nome}")
    print("Total de registros:", len(df))
    print("Alunos únicos:", df["aluno_id"].nunique())
    print(
        "Registros duplicados por aluno e fase:",
        df.duplicated(["aluno_id", "fase"]).sum()
    )

In [ ]:
for nome in ["pareado_vocabulario", "pareado_tde"]:

    if nome not in bases:
        continue

    df = bases[nome]

    print(f"\nBASE: {nome}")

    print("Pré-teste ausente:", df["total_pre"].isna().sum())
    print("Pós-teste ausente:", df["total_pos"].isna().sum())

    completos = df["total_pre"].notna() & df["total_pos"].notna()

    print("Registros completos:", completos.sum())

In [ ]:
for nome in ["pareado_vocabulario", "pareado_tde"]:

    if nome not in bases:
        continue

    df = bases[nome]

    print(f"\nBASE: {nome}")

    display(
        df.groupby("fase")
        .agg(
            registros=("aluno_id", "size"),
            alunos_unicos=("aluno_id", "nunique")
        )
    )

## Passo 2.4: Identificação das Variáveis Disponíveis.

In [ ]:
for nome, df in bases.items():

    print(f"\n{nome}")
    for coluna in df.columns:
        print("-", coluna)

In [ ]:
termos = [
    "aluno",
    "escola",
    "turma",
    "serie",
    "fase",
    "pre",
    "pos",
    "total",
    "mae",
    "pai",
    "responsavel",
    "familia",
    "idade"
]

for nome, df in bases.items():
    encontradas = [
        coluna for coluna in df.columns if any(termo in coluna.lower() for termo in termos)
    ]

    print(f"\n{nome}:")
    print(encontradas)

In [ ]:
for nome, df in bases.items():

    print(f"\nBASE: {nome}")

    colunas_categoricas = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for coluna in colunas_categoricas:
        print(f"\n{coluna}")
        print(df[coluna].value_counts(dropna=False).head(10))

## Passo 2.5: Análise da Qualidade e Cobertura dos Dados.

In [ ]:
for nome, df in bases.items():

    cobertura = pd.DataFrame({
        "preenchidos": df.notna().sum(),
        "ausentes": df.isna().sum(),
        "cobertura_percentual": (
            df.notna().mean() * 100
        ).round(2)
    })

    print(f"\nBASE: {nome}")
    display(cobertura)

In [ ]:
for nome, df in bases.items():

    print(f"\nBASE: {nome}")

    print("Linhas duplicadas completas:", df.duplicated().sum())

    if {"aluno_id", "fase"}.issubset(df.columns):
        print(
            "Duplicidades aluno + fase:",
            df.duplicated(["aluno_id", "fase"]).sum()
        )

In [ ]:
for nome, df in bases.items():

    print(f"\nBASE: {nome}")

    for coluna in ["fase", "serie", "turma", "escola"]:

        if coluna in df.columns:
            print(f"\n{coluna}")
            print(df[coluna].value_counts(dropna=False))

In [ ]:
for nome, df in bases.items():

    print(f"\nBASE: {nome}")

    colunas = [
        coluna
        for coluna in ["total_pre", "total_pos"]
        if coluna in df.columns
    ]

    if colunas:
        display(df[colunas].describe())

## Verificações.

> 1. Registros pareados não significam necessariamente avaliações completas.

In [ ]:
for nome in ["pareado_vocabulario", "pareado_tde"]:

    if nome not in bases:
        continue

    df = bases[nome]

    completos = (
        df["completo_pre"].eq(True) & df["completo_pos"].eq(True)
    )  

    print(f"\nBASE: {nome}")
    print("Registros pareados:", len(df))
    print("Pré completo:", df["completo_pre"].sum())
    print("Pós completo:", df["completo_pos"].sum())
    print("Pré e pós completos:", completos.sum())
    print("Pelo menos uma avaliação incompleta:", (~completos).sum())

    print("\nCompletude por fase:")

    display(
        df.assign(par_completo=completos)
        .groupby("fase")
        .agg(
            registros=("aluno_id", "size"),
            pares_completos=("par_completo", "sum")
        )
    )

> 2. A verificação de duplicidades precisa ser interpretada corretamente.

In [ ]:
for nome in ["dataset_vocabulario", "dataset_tde"]:

    df = bases[nome]

    print(f"\nBASE: {nome}")

    print(
        "Duplicidades por aluno, fase e período:",
        df.duplicated(
            ["aluno_id", "fase", "periodo"]
        ).sum()
    )

> 3. A cobertura de dim_aluno precisa ser investigada em relação às bases pareadas.

In [ ]:
dim = bases["dim_aluno"]

for nome in ["pareado_vocabulario", "pareado_tde"]:

    df = bases[nome]

    alunos_pareados = set(df["aluno_id"].dropna())
    alunos_dim = set(dim["aluno_id"].dropna())

    encontrados = alunos_pareados & alunos_dim
    ausentes = alunos_pareados - alunos_dim

    print(f"\nBASE: {nome}")
    print("Alunos únicos:", len(alunos_pareados))
    print("Encontrados em dim_aluno:", len(encontrados))
    print("Sem correspondência:", len(ausentes))
    print(
        "Cobertura:",
        round(len(encontrados) / len(alunos_pareados) * 100, 2),
        "%"
    )

## Conclusão da Investigação e Seleção de Dados

A auditoria das bases para a etapa de modelagem preditiva confirmou:

1. **Bases Selecionadas**: Utilizaremos `pareado_vocabulario` e `pareado_tde` como fontes primárias de desempenho temporal (pré e pós-teste), integradas com `dim_aluno` para características contextuais.
2. **Registros Elegíveis**: Apenas avaliações com completude simultânea no pré e pós-teste (`completo_pre == True & completo_pos == True`) foram selecionadas para garantir que o ganho de pontuação seja medido sobre instrumentos respondidos validamente (>=80% dos itens):
   - **Vocabulário**: 3.320 pares completos (1.989 estudantes únicos).
   - **TDE II**: 3.796 pares completos (2.069 estudantes únicos).
3. **Target**: `ganho = total_pos - total_pre`. Ambas as bases possuem distribuição unimodal sem valores nulos ou infinitos.
4. **Cobertura Demográfica**: `dim_aluno` cobre 99,07% dos pares completos em Vocabulário e 99,42% em TDE II.
5. **Prevenção de Vazamento**: As variáveis futuras (`total_pos`, `pct_pos`, `completo_pos`) e dados identificáveis (`aluno_id`, `responsavel_legal`) foram estritamente excluídos das features preditoras.
6. **Estratégia de Validação**: Divisão de treino/teste por grupos baseada em `aluno_id` para impedir que o mesmo aluno apareça em ambos os conjuntos.